<a href="https://colab.research.google.com/github/romesalarda/comp3226-password-manager-extension/blob/marianna/ecg-arrhythmia-v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Single-Lead ECG Arrhythmia Classification**



Arrhythmia Classification, 5 classes
======================================================

Based on: Md Rabiul Islam – Single Lead Arrhythmia Classification (2023)

Author: Marianna Kottas

Institution: University of Southampton

Programme: BSc Computer Science

Project: Enhancing Patient-Facing Interpretability in an ECG Arrhythmia Classifier

Model: CNN_LSTM with Attention


Dataset
-------
MIT-BIH Arrhythmia Database

• 48 ECG recordings from 47 patients  
• Sampling frequency: 360 Hz  
• Recording length: ~30 minutes per patient  


Task
----
Heartbeat classification into the 5 AAMI categories:

0 – Normal (N)  
1 – Supraventricular ectopic beat (S)  
2 – Ventricular ectopic beat (V)  
3 – Fusion beat (F)  
4 – Unknown / unclassifiable beat (Q)


Pipeline
--------
1. ECG signal loading
2. Wavelet denoising
3. R-peak extraction (using dataset annotations)
4. Heartbeat segmentation (300 samples per beat)
5. Class grouping (15 → 5 AAMI classes)
6. Training set balancing (undersampling + SMOTE)
7. CNN-LSTM classifier
8. Evaluation (accuracy, confusion matrix, per-class metrics)
9. Inference pipeline (prediction + confidence + clinical risk)


Model
-----
CNN-LSTM hybrid neural network implemented in TensorFlow / Keras.


Output
------
Prediction pipeline producing structured output:

{
    "prediction": class_label,
    "confidence": probability_score,
    "risk_level": clinical_risk_category,
    "probabilities": class_probability_vector
}


Purpose
-------
The system is designed for integration into a patient-facing ECG
interpretation dashboard that communicates model predictions,
confidence, and clinical risk in an accessible format.

## **a. Summary, At a Glance**


*   **Task:** Arrhythmia Classification, 5 classes
*   **Dataset:** MIT-BIH ECG dataset, Dataset website: [Here](https://physionet.org/content/mitdb/1.0.0/)
*   **Model:** CNN_LSTM with Attention

## **b. Information about Dataset:**


The dataset can be downloaded from the original [website](https://physionet.org/content/mitdb/1.0.0/) or by clicking the [link](https://physionet.org/static/published-projects/mitdb/mit-bih-arrhythmia-database-1.0.0.zip) directly. The overall information about the dataset is given [here](https://archive.physionet.org/physiobank/database/html/mitdbdir/intro.htm).
1. The MIT-BIH dataset contains **48 readings**.
2. Number of **patients = 47** (25 Men, 22 Women). Only 1 patient has 2 readings. Rest 1 readings/patients.
3. Each reading contains a. ECG signals b. Annotations
4. ECG signals length = **30 Minutes** (or slightly over). Sampling Frequency = **360 Hz**.
5. Each reading contains **2 Lead ECG** signals that are Modified Limb Lead II and Modified Lead V1 (Occationally V2, V5, V4 just only once). Simply, Two Leads: **MLII, V1**.
6. Annotation is given for each beat. Although 20 types of beats are annotated in dataset. We consider only **15 types** of beats grouped in **5 classes** followed by the recommendation of **AAMI**.
* N - Normal
* S - Supraventricular premature beat
* V - Premature ventricular contraction
* F - Fusion of ventricular and normal beat
* Q - Unclassifiable / Unknown beat

### **Additional Dataset Information:**
The source of the ECGs included in the MIT-BIH Arrhythmia Database is a set of over **4000 long-term Holter recordings** that were obtained by the **Beth Israel Hospital** Arrhythmia Laboratory between **1975 and 1979**. Approximately **60%** of these recordings were obtained from **inpatients**. The subjects were 25 men aged 32 to 89 years, and 22 women aged 23 to 89 years

# **Main Parts of the Study:**

1.   **Part A: Installing Packages and Basic Visualization of ECG**
2.   **Part B: Denoising, R-Peak Detection, Segmentation**
3.   **Part C: Dataset Loading**
4.   **Part D: Train-Test Splitting and Class Balancing**
5.   **Part E: Model Building and Training**
6.   **Part F: Results**

# **Part A: Installing Packages and Basic Visualization of ECG**

## **A0: Mount Drive**

In [1]:
# Mount Google Drive + Model Path

from google.colab import drive
import os

# Mount your Drive
drive.mount('/content/drive')

# Model location (you already created this folder)
MODEL_DIR = "/content/drive/MyDrive/ecg_model"
MODEL_PATH = MODEL_DIR + "/ecg_model.keras"

# Safety: ensure folder exists
os.makedirs(MODEL_DIR, exist_ok=True)

print("Model path:", MODEL_PATH)


Mounted at /content/drive
Model path: /content/drive/MyDrive/ecg_model/ecg_model.keras


## **A1: Installing Packages**

In [2]:
# wfdb is not normally installed in Colab
!pip install wfdb
# For interactive diagram
!pip install neurokit2 plotly

!pip install -q "pandas==2.2.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 35.1 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.

In [ ]:
import os, wfdb

BASE = "/content/data"
MIT = os.path.join(BASE, "mitdb")
os.makedirs(MIT, exist_ok=True)

wfdb.dl_database("mitdb", dl_dir=MIT)

project_path = MIT + "/"
DATA_ROOT = MIT
RECORDS = os.path.join(MIT, "RECORDS")

FIG_DIR = "/content/figures/"
os.makedirs(FIG_DIR, exist_ok=True)

Generating record list for: 100
Generating record list for: 101
Generating record list for: 102
Generating record list for: 103
Generating record list for: 104
Generating record list for: 105
Generating record list for: 106
Generating record list for: 107
Generating record list for: 108
Generating record list for: 109
Generating record list for: 111
Generating record list for: 112
Generating record list for: 113
Generating record list for: 114
Generating record list for: 115
Generating record list for: 116
Generating record list for: 117
Generating record list for: 118
Generating record list for: 119
Generating record list for: 121
Generating record list for: 122
Generating record list for: 123
Generating record list for: 124
Generating record list for: 200
Generating record list for: 201
Generating record list for: 202
Generating record list for: 203
Generating record list for: 205
Generating record list for: 207
Generating record list for: 208
Generating record list for: 209
Generati

In [ ]:
# Importing packages
import os
import datetime
import wfdb
import pywt
import seaborn
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from os.path import join as osj
from collections import Counter
import pandas as pd

## **A2: Basic Visualization of ECG**
Basic code for loading (reading), plotting and playing with ECG signals

### **a. Getting Recordings' IDs**
The ECG recordings are named after Patients' IDs (from 100 to 234), sorted but not consecutive. Total 48 recordings.

In [ ]:
import wfdb

patient_ids = wfdb.get_record_list("mitdb")
patient_ids

### **b. 1 Patient ECG loading and plotting**
Extracting 2 leads ECG signals of a patient (for example: 100), and saving in two lists.

In [ ]:
#Extracting just 1 patient ECG signal and info
lead0 = {}  # without this it shows lead0[100] is not defined
lead1 = {}
patient_id = 100
signals, info = wfdb.io.rdsamp(osj(DATA_ROOT, str(100)))
lead0[100] = signals[:, 0]
lead1[100] = signals[:, 1]

In [ ]:
# Visualization of 1 patients signal and info
print(type(lead0[100]))
print(lead0[100].shape)
plt.plot(lead0[100])
print(info)

In [ ]:
# ECG signal per second
a = lead0[100][0: 3000]
plt.figure(figsize=(12, 4), dpi=90)
plt.plot(a)

### **c. All patients' ECG loading**

In [ ]:
# Loading all patients ECG SIGNALs using for loop
def get_ecg_signals(patient_ids):
    lead0 = {}
    lead1 = {}
    for id_ in patient_ids:
        signals, info = wfdb.io.rdsamp(osj(DATA_ROOT, str(id_)))
        lead0[id_] = signals[:, 0]
        lead1[id_] = signals[:, 1]
        print(f'Signal of patient {id_} extracted')
    return lead0, lead1

In [ ]:
# Loading all patient ECG INFORMATION
def get_ecg_info(patient_ids):
    _, info = wfdb.io.rdsamp(osj(DATA_ROOT, str(patient_ids)))
    resolution = 2**11  # Number of possible signal values we can have.
    info["resolution"] = 2**11
    return info

In [ ]:
lead0, lead1 = get_ecg_signals(patient_ids)

In [ ]:
# Plot any patient signal from any time frame
patient_id = "100" # can change
starting_time = 0 # can change
ending_time = 10 # can change

# Scaling
starting_signal_point = starting_time*350
ending_signal_point = ending_time*350 # As sampling frequency is 350 Hz
x = np.arange(starting_time, ending_time, 1/350)
signal = lead0[patient_id][starting_signal_point: ending_signal_point]

plt.figure(figsize=(12, 3), dpi=100)
plt.plot(x, signal)
plt.title(f'ECG signla of patient {patient_id}')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude (mV)')

In [ ]:
# ECG info of any patient
ecg_info = get_ecg_info(patient_ids[0])
ecg_info

# **Part B: Denoising, R-Peak Detection, Segmentation**

### **B1: Denoising**
Noise removing by using Discrete Wavelet Transform (DCT)

In [ ]:
# User defined fucntion for DWT and reconstruction
def denoise(data):
    # wavelet transform
    coeffs = pywt.wavedec(data=data, wavelet='db5', level=9)
    cA9, cD9, cD8, cD7, cD6, cD5, cD4, cD3, cD2, cD1 = coeffs

    # Threshold denoising
    threshold = (np.median(np.abs(cD1)) / 0.6745) * (np.sqrt(2 * np.log(len(cD1))))
    cD1.fill(0)
    cD2.fill(0)
    for i in range(1, len(coeffs) - 2):
        coeffs[i] = pywt.threshold(coeffs[i], threshold)

    # Inverse wavelet transform to obtain the denoised signal
    rdata = pywt.waverec(coeffs=coeffs, wavelet='db5')
    return rdata

In [ ]:
# Ploting a signal before denoising
record = wfdb.rdrecord(project_path + '100', channel_names=['MLII'])
data = record.p_signal.flatten()
plt.plot(data[0:500])

In [ ]:
# Same signal after denoising
rdata = denoise(data=data)
plt.plot(rdata[0:500])

### **B2: R-Peak Detection**
R-peak is annotated in MIT-BIH dataset. Just need to read the file.

In [ ]:
# For exmaple, we extract '100' recording annotation
annotation = wfdb.rdann(project_path + '100', 'atr')
Rlocation = annotation.sample
print(Rlocation)
Rclass = annotation.symbol
print(Rclass)

In [ ]:
len(annotation.symbol)

In [ ]:
# R-peak ploting
x = np.arange(1, 1081)

n_peak =5
r_peak_x = []
r_peak_y = []
for i in range(0, n_peak):
  r_peak_x.append(Rlocation[i])
  r_peak_y.append(rdata[Rlocation[i]])

plt.plot(x, data[0:1080], color='red')
plt.scatter(r_peak_x, r_peak_y)

### **B3: Segmentation**
Each ECG signal is segmented by using a window **length of 300**. From R-peak location, **99** samples taken from **left** and **201** samples from **right**. Thus a complete **heartbeat** is found.

In [ ]:
# Plotting 3 heartbeats
k = np.arange(100, 103)
for i in k:
  # print(i)
  # print(Rlocation[i] - 99, Rlocation[i] + 201)
  x_train = rdata[Rlocation[i] - 99:Rlocation[i] + 201]
  plt.plot(x_train)
  print(x_train.shape)
plt.show()

### **B4: Complete Preprocessing Figures**
The complete preprocessing including denosinsing, R-peak location detection and segmentation is expected to view in a single figure.

In [ ]:
r_peak_xx = Rlocation[0], Rlocation[1], Rlocation[2], Rlocation[3]
r_peak_yy = rdata[Rlocation[0]], rdata[Rlocation[1]], rdata[Rlocation[2]], rdata[Rlocation[3]]

In [ ]:
# Plotting R-peaks and segmentation lines
fig = plt.figure(figsize=(10,3), dpi=600)
n_peak =5
r_peak_x = []
r_peak_y = []
for i in range(0, n_peak):
  r_peak_x.append(Rlocation[i])
  r_peak_y.append(rdata[Rlocation[i]])
x = np.arange(1, 1081)
plt.plot(x, rdata[0: 1080], color='red')
plt.scatter(r_peak_x, r_peak_y)

# line plotting
plt.axvline(x = Rlocation[2], color = 'k', linestyle = ':')
plt.axvline(x = Rlocation[2]-99, color = 'k', linestyle = '--')
plt.axvline(x = Rlocation[2]+201, color = 'k', linestyle = '--')

In [ ]:
# Plot together raw, denoised and segmted signal
fig = plt.figure(figsize=(10,9), dpi=600)
x = np.arange(1, 1081)

# Raw signal plotting
plt.subplot(3, 1, 1)
plt.plot(x/360, data[0:1080], color='red')
plt.xlabel('Time (s)')
plt.ylabel('Voltage (mV)')
plt.title('Raw ECG signal')

# Denoised signal plotting
plt.subplot(3, 1, 2)
plt.plot(x/360, rdata[0:1080], color='red')
plt.title('Denoised ECG signal')

# Segmentation visualization using two border lines
plt.subplot(3, 1, 3)
n_peak =5
r_peak_x = []
r_peak_y = []
for i in range(0, n_peak):
  r_peak_x.append(Rlocation[i])
  r_peak_y.append(rdata[Rlocation[i]])
x = np.arange(1, 1081)
plt.plot(x, rdata[0: 1080], color='red')
plt.scatter(r_peak_x, r_peak_y)
# line plotting
plt.axvline(x = Rlocation[2], color = 'k', linestyle = ':') # 3rd r-peak
plt.axvline(x = Rlocation[2]-99, color = 'k', linestyle = '--')
plt.axvline(x = Rlocation[2]+201, color = 'k', linestyle = '--')

plt.xlabel('# Sample')
plt.ylabel('Voltage (mV)')
plt.title('Segmentation using 3rd R-peak')

plt.subplots_adjust(left=0.1,
                    bottom=0.1,
                    right=0.9,
                    top=0.9,
                    wspace=0.4,
                    hspace=0.4)

fig.savefig(FIG_DIR + "Denoised_and_segmented_ECG.png")

# **Part C: Dataset Loading**

## **C1: Loading whole data**

In [ ]:
# Read ECG signals and corresponding label
def getDataSet(number, X_data, Y_data):

    # Considering 15 types ECG heartbeats that are later grouped in 5 classes
    ecgClassSet = ['N', 'L', 'R', 'e', 'j', 'A', 'a', 'J', 'S', 'V', 'E', 'F', '/', 'f', 'Q']

    # Reading Channel names
    _, info = wfdb.io.rdsamp(osj(project_path, number))
    channels = info['sig_name']
    channel1, channel2 = channels[0], channels[1]
    print(channel1, channel2)


    # Read ECG data records
    print("reading " + number+ " ECG data...")
    record = wfdb.rdrecord(project_path + number, channel_names=[channel1])
    data = record.p_signal.flatten()
    rdata = denoise(data=data)

    # Obtain the position and corresponding label of the R wave in the ECG data record
    annotation = wfdb.rdann(project_path + number, 'atr')
    Rlocation = annotation.sample
    Rclass = annotation.symbol

    # Unstable data before and after removal
    start = 2  # if it creates problem then except will do the job
    end = 3
    i = start
    j = len(annotation.symbol) - end

    # Making labels, Y_data Convert NSVFQ in order to 0123456...14
    while i < j:
        try:
            beat_type = Rclass[i]
            lable = ecgClassSet.index(beat_type)  # when beat is like '+' or other it will go on except loop
            x_train = rdata[Rlocation[i] - 99:Rlocation[i] + 201]
            X_data.append(x_train)
            Y_data.append(lable)
            i += 1
        except ValueError:
            # print(f' when i = {i}, beat type is out of our choise. For example +, [, ! or other')
            i += 1
    return X_data, Y_data

In [ ]:
# Load the dataset and preprocess it
def loadData():
    numberSet = ['100', '101', '102', '103', '104', '105', '106', '107', '108', '109',
                 '111', '112', '113', '114', '115', '116', '117', '118', '119', '121',
                 '122', '123', '124', '200', '201', '202', '203', '205', '207', '208',
                 '209', '210', '212', '213', '214', '215', '217', '219', '220', '221',
                 '222', '223', '228', '230', '231', '232', '233', '234'] # 48 readings
    dataSet = []
    lableSet = []
    for n in numberSet:
        # getDataSet(n, dataSet, lableSet)
        dataSet, lableSet = getDataSet(n, dataSet, lableSet)

    # Turn numpy array, scramble the order
    dataSet = np.array(dataSet).reshape(-1, 300)
    lableSet = np.array(lableSet).reshape(-1, 1)
    train_ds = np.hstack((dataSet, lableSet))
    np.random.shuffle(train_ds)

    # dataset and its label set
    X = train_ds[:, :300]
    Y = train_ds[:, 300]
    return X, Y

In [ ]:
# Input X and Output Y data loading
X, Y = loadData()

In [ ]:
# Counting the number of each type of heartbeats
Y_list = list(Y)
Counter(Y_list)

## **C2: Ploting 15 Different Heartbeats**

In [ ]:
# making pandas dataframe
df_X = pd.DataFrame(X)
df_Y = pd.DataFrame(Y)

In [ ]:
# changing the name from 0 to 300
df_Y.rename(columns = {0:300}, inplace = True)
# join X and Y
df = pd.concat([df_X, df_Y], axis=1)

In [ ]:
def Plot_Random_Beat(type, num):

  ecgClassSet = ['N', 'L', 'R', 'e', 'j', 'A', 'a', 'J', 'S', 'V', 'E', 'F', 'slash', 'f', 'Q']

  ecgClassName = ['Normal (N)', 'Left bundle br. bl. (L)', 'Right bundle br. bl. (R)',
                  'Atrial escape (e)', 'Nodal jun. esc. (j)', 'Atrial premature (A)',
                  'Aberrated atrial prem. (a)', 'Nodal jun. pre. (J)',
                  'Supraventricular prem. (S)', 'Premature ventr. (V)',
                  'Ventricular escape (E)', 'Fusion of ve. & no. (F)',
                  'Paced (/)', 'Fusion of pa. & no. (f)',
                  'Unclassifiable(Q)']

  # getting only a specific class ECG signal
  df_0 = df.loc[df[300]==type]  # For normanl class: 0, shape is 74920,301
  df_0 = df_0.drop(columns=[300]) # changing the shape to 74920,300

  # selecting some random row to plot
  if num<=df_0.shape[0]:
    np.random.seed(234)
    random_beat_number = np.random.randint(df_0.shape[0], size=(num))
    random_beat_number = list(random_beat_number)
  else: # Needed for Supraventricular Premature Beat (S) only, as it contains only 2 beats
    print(f"Warning: You have only {df_0.shape[0]} beat, but asked to plot {num}")
    random_beat_number = np.arange(0, df_0.shape[0])
    random_beat_number = list(random_beat_number)

  # ploting the ECG signal
  for i in random_beat_number:
    ecg_beat = df_0.iloc[i]
    plt.plot(ecg_beat)
  plt.title(str(ecgClassName[type]))

In [ ]:
# Plotting 15 different types of heartbeat
fig = plt.figure(figsize=(16,7), dpi=400)
fig.tight_layout(pad=15.0)
for i in range(15):
  plt.subplot(3,5,i+1)
  plt.subplots_adjust(left=0.1,
                    bottom=0.1,
                    right=0.9,
                    top=0.9,
                    wspace=0.4,
                    hspace=0.4)
  Plot_Random_Beat(type=i, num=10)

fig.savefig(FIG_DIR + "all_heartbeats.png")

# **Part D: Train-Test Splitting and Class Balancing**

## **D1: Data loading**
Data is already loaded, **this step can be skipped.** However, here the whole dataset is saved in a train_ds variable.

### **a. Load whole data**

In [ ]:
# Load the dataset and preprocess it
def loadData():
    numberSet = ['100', '101', '102', '103', '104', '105', '106', '107', '108', '109',
                 '111', '112', '113', '114', '115', '116', '117', '118', '119', '121',
                 '122', '123', '124', '200', '201', '202', '203', '205', '207', '208',
                 '209', '210', '212', '213', '214', '215', '217', '219', '220', '221',
                 '222', '223', '228', '230', '231', '232', '233', '234']  # 48 readings
    dataSet = []
    lableSet = []
    for n in numberSet:
        # getDataSet(n, dataSet, lableSet)
        dataSet, lableSet = getDataSet(n, dataSet, lableSet)

    # Turn numpy array, scramble the order
    dataSet = np.array(dataSet).reshape(-1, 300)
    lableSet = np.array(lableSet).reshape(-1, 1)
    train_ds = np.hstack((dataSet, lableSet))
    np.random.shuffle(train_ds)
    return train_ds

In [ ]:
# Load the whole dataset (109305,301). Each row indicate an ECG beat time series data upto 300
# and 301 colum is its label among 15 difference level
train_ds = loadData()

In [ ]:
Y = train_ds[:, 300]

In [ ]:
# Here 15 class of ECG data are saved
Y_list = list(Y)
Counter(Y_list)

### **b. 15 types to 5 level conversion**

In [ ]:
# 15 level to 5 level conversion
Y_5class = np.copy(Y)

for i in range(Y.shape[0]):
  # print(i)
  if 0 <= Y[i] <= 4:
    Y_5class[i] = 0
  if 5 <= Y[i] <= 8:
    Y_5class[i] = 1
  if 9 <= Y[i] <= 10:
    Y_5class[i] = 2
  if Y[i] == 11:
    Y_5class[i] = 3
  if 12 <= Y[i] <= 14:
    Y_5class[i] = 4
print('changing done')

In [ ]:
Y_5class_list = list(Y_5class)
Counter(Y_5class_list)

In [ ]:
ecg_dataset = np.copy(train_ds)

In [ ]:
# label encode the target variable # just convert numpy.float64 to numpy.int64
from sklearn.preprocessing import LabelEncoder
Y_5class = LabelEncoder().fit_transform(Y_5class)

In [ ]:
ecg_data = ecg_dataset[:, :300]
ecg_lable = Y_5class.reshape(-1, 1) # otherwise np.hstack will not work

In [ ]:
# Complete ECG dataset with 5 type of Arrhythmia
ecg_dataset_5 = np.hstack((ecg_data, ecg_lable))

### **c. Per class data status checking (Full data)**

In [ ]:
# Convert ndarray to dataframe
df_ecg = pd.DataFrame(ecg_dataset_5)
class_data = df_ecg[300].value_counts()
class_data

In [ ]:
# per class data status plotting,
plt.bar(class_data.index, class_data.values, color ='maroon')
plt.show()

In [ ]:
# shortcut for per class data status plotting,
# Order is not maintained by class. Higher to lower
df_ecg[300].value_counts().plot(kind='bar')

## **D2: Train-Test Spliting**
**Note: Class Balance should be done on Training Data Only. Not Testing Data.**

In [ ]:
# train test splitting
from sklearn.model_selection import train_test_split
ecg_data = ecg_dataset_5[:, :300]
ecg_label = ecg_dataset_5[:, 300]
x_train, x_test, y_train, y_test = train_test_split(ecg_data, ecg_label,
                                   random_state=104,
                                   test_size=0.20,
                                   shuffle=True)

In [ ]:
# reshaping for using hstack function
y_train = y_train.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)
train_data = np.hstack((x_train, y_train))
test_data = np.hstack((x_test, y_test))

In [ ]:
#  converting dataframe
train_data = pd.DataFrame(train_data)
test_data = pd.DataFrame(test_data)

In [ ]:
# saving the test data (in imbalanced condition)
file_name = project_path + 'test_data.pkl'
test_data.to_pickle(file_name)

**Training dataset status checking:** balanced / imbalanced

In [ ]:
# Imblanced training data graph ploting
class_data = train_data[300].value_counts()
print(class_data)
plt.bar(class_data.index, class_data.values, color ='maroon')
plt.show()

## **D3: Class balancing by undersampling and SMOTE**
**SMOTE** stands for '**Synthetic Minority Oversampling Technique**'.
Plan for train data
1. Class 1: Randomly selected 50000 data
2. Class 1, 2, 3, 4: Use SMOTE to oversample upto 50000 data

In [ ]:
# extracting class 0 and 4 others class
train_data_0 = train_data.loc[(train_data[300] == 0)]
train_data_1234 = train_data.loc[(train_data[300] != 0)]

In [ ]:
# 1. Class 1: Randomly selected 50000 data
from sklearn.utils import resample
train_data_0_resampled=train_data_0.sample(n=50000,random_state=42)

# convert dataframe to numpy array
train_data_0_resampled = train_data_0_resampled.to_numpy()

In [ ]:
# 2. Class 1, 2, 3, 4: Use SMOTE to oversample upto 50000 data

# converting from df to np ndarray
train_data_1234_arr = train_data_1234.to_numpy()
X_4cl, y_4cl = train_data_1234_arr[:, :-1], train_data_1234_arr[:, -1]

from imblearn.over_sampling import SMOTE
# transform the dataset
strategy = {1:50000, 2:50000, 3:50000, 4:50000}
oversample = SMOTE(sampling_strategy=strategy)
X, y = oversample.fit_resample(X_4cl, y_4cl)

y = y.reshape(-1, 1)
train_data_1234_resampled = np.hstack((X, y))

In [ ]:
# Join the class 0 and 1234
train_data_resampled = np.vstack((train_data_0_resampled, train_data_1234_resampled))

# shuffle the data, needed for proper training
np.take(train_data_resampled,np.random.permutation(train_data_resampled.shape[0]),axis=0,out=train_data_resampled)

In [ ]:
# blanced training data graph ploting
train_data_r = pd.DataFrame(train_data_resampled)
class_data = train_data_r[300].value_counts()
print(class_data)
plt.bar(class_data.index, class_data.values, color ='maroon')
plt.show()

# save balanced training data
file_name = project_path + 'train_data_SMOTE.pkl'
train_data_r.to_pickle(file_name)

In [ ]:
data_bal = np.array(class_data)
data_bal2 = data_bal.reshape(1, 5)

In [ ]:
# a single plot which gives proper illustration before and after class balancing
import seaborn as sns
sns.set()
sns.color_palette("hls", 8)

fig = plt.figure(figsize=(7,4), dpi=600)
plt.subplot(121)
sns.barplot(x = ['N', 'S', 'V', 'F', 'Q'], y = [72420, 2212, 5774, 637, 6401])
plt.ylim(0, 75000)
plt.title('Training Data, Imbalanced')

plt.subplot(122)
sns.barplot(x = ['N', 'S', 'V', 'F', 'Q'], y = class_data.values)
plt.ylim(0, 75000)
plt.title('Balanced by SMOTE')

plt.subplots_adjust(left=0.1,
                    bottom=0.1,
                    right=0.9,
                    top=0.9,
                    wspace=0.4,
                    hspace=0.5)

fig.savefig(FIG_DIR + "Class_balancing.png")

# **Part E: Model Building and Training**
A **CNN-LSTM and attention** based hybrid model is formulated.

## **E1: Attention Mechanism**
**Convolutional Block Attention Module** ([CBAM](https://arxiv.org/abs/1807.06521)) consists two parts (i) **channel attention**, (ii) spatial attention. ECG is an 1D signal and 1-lead ECG is used in the modle.Therefore, only channel attention is only used.

In [ ]:
# udf for channel attention mechanism
class ChannelAttention(tf.keras.layers.Layer):
      def __init__(self, filters, ratio):
        super(ChannelAttention, self).__init__()
        self.filters = filters
        self.ratio = ratio

        def build(self, input_shape):
            self.shared_layer_one = tf.keras.layers.Dense(self.filters//self.ratio,
                             activation='relu', kernel_initializer='he_normal',
                              use_bias=True,
                              bias_initializer='zeros')
            self.shared_layer_two = tf.keras.layers.Dense(self.filters,
                             kernel_initializer='he_normal',
                             use_bias=True,
                             bias_initializer='zeros')

        def call(self, inputs):
            # AvgPool
            avg_pool = tf.keras.layers.GlobalAveragePooling1D()(inputs)


            avg_pool = self.shared_layer_one(avg_pool)
            avg_pool = self.shared_layer_two(avg_pool)

            # MaxPool
            max_pool = tf.keras.layers.GlobalMaxPooling1D()(inputs)
            max_pool = tf.keras.layers.Reshape((1,1,filters))(max_pool)

            max_pool = shared_layer_one(max_pool)
            max_pool = shared_layer_two(max_pool)


            attention = tf.keras.layers.Add()([avg_pool,max_pool])
            attention = tf.keras.layers.Activation('sigmoid')(attention)

            return tf.keras.layers.Multiply()([inputs, attention])

In [ ]:
# udf for spatial attention mechanism
class SpatialAttention(tf.keras.layers.Layer):
      def __init__(self, kernel_size):
        super(SpatialAttention, self).__init__()
        self.kernel_size = kernel_size

        def build(self, input_shape):
            self.conv2d = tf.keras.layers.Conv2D(filters = 1,
                    kernel_size=self.kernel_size,
                    strides=1,
                    padding='same',
                    activation='sigmoid',
                    kernel_initializer='he_normal',
                    use_bias=False)

        def call(self, inputs):

            # AvgPool
            avg_pool = tf.keras.layers.Lambda(lambda x: tf.keras.backend.mean(x, axis=3, keepdims=True))(inputs)

            # MaxPool
            max_pool = tf.keras.layers.Lambda(lambda x: tf.keras.backend.max(x, axis=3, keepdims=True))(inputs)

            attention = tf.keras.layers.Concatenate(axis=3)([avg_pool, max_pool])

            attention = self.conv2d(attention)


            return tf.keras.layers.multiply([inputs, attention])

## **E2: CNN-LSTM and attention model architecture**

### **a. Model building**

In [ ]:
# Build a CNN model
def buildModel():
    newModel = tf.keras.models.Sequential([
        tf.keras.layers.InputLayer(input_shape=(300, 1)),
        # The first convolutional layer, sixteen 21x1 convolution kernels
        tf.keras.layers.Conv1D(filters=16, kernel_size=21, strides=1, padding='same', activation='relu'),
        ChannelAttention(16, 8),
        # SpatialAttention(7),
        # The first pooling layer, max pooling, 3x1 convolution kernels, stride 2
        tf.keras.layers.MaxPool1D(pool_size=3, strides=2, padding='same'),
        # The second convolution layer, 32 23x1 convolution kernels
        tf.keras.layers.Conv1D(filters=32, kernel_size=23, strides=1, padding='same', activation='relu'),
        ChannelAttention(32, 8),
        # SpatialAttention(7),
        # The second pooling layer, max pooling, 3x1 convolution kernels, with a stride of 2
        tf.keras.layers.MaxPool1D(pool_size=3, strides=2, padding='same'),
        # The third convolution layer, 64 25x1 convolution kernels
        tf.keras.layers.Conv1D(filters=64, kernel_size=25, strides=1, padding='same', activation='relu'),
        ChannelAttention(64, 8),
        # SpatialAttention(7),
        # The third pooling layer, average pooling, 3x1 convolution kernels, stride 2
        tf.keras.layers.AvgPool1D(pool_size=3, strides=2, padding='same'),
        # The fourth convolution layer, 128 27x1 convolution kernels
        tf.keras.layers.Conv1D(filters=128, kernel_size=27, strides=1, padding='same', activation='relu'),
        ChannelAttention(128, 8),
        SpatialAttention(7),
        # LSTM layer, 64 nodes
        tf.keras.layers.LSTM(64, return_sequences=True),
        # Dropout layer,dropout = 0.2
        tf.keras.layers.Dropout(rate=0.2),
        # LSTM layer, 32 nodes
        tf.keras.layers.LSTM(32, return_sequences=True),
        # Flatten the layer to facilitate the processing of the fully connected layer
        tf.keras.layers.Flatten(),
        # Fully connected layer, 128 nodes
        tf.keras.layers.Dense(128, activation='relu'),
        # Dropout layer,dropout = 0.2
        tf.keras.layers.Dropout(rate=0.2),
        # Fully connected layer, 5 nodes
        tf.keras.layers.Dense(5, activation='softmax')
    ])
    return newModel

### **b. Hyperparameter Tuning**
Many architectures of model is found by changing hyperparameters.
* Model: with or without channel or spatial attention.
* CNN Filters: Number of filters in each Conv layer is changing like 4, 16, 32, 64, 128 etc.
* LSTM Units: Number of units of two LSTM layer is varing like 32, 64, 128 etc.

In [ ]:
# Build a CNN model
def buildModel():
    newModel = tf.keras.models.Sequential([
        tf.keras.layers.InputLayer(input_shape=(300, 1)),
        # The first convolutional layer, four 21x1 convolution kernels
        tf.keras.layers.Conv1D(filters=16, kernel_size=21, strides=1, padding='same', activation='relu'),
        # The first pooling layer, max pooling, four 3x1 convolution kernels, stride 2
        tf.keras.layers.MaxPool1D(pool_size=3, strides=2, padding='same'),
        # The second convolution layer, 16 23x1 convolution kernels
        tf.keras.layers.Conv1D(filters=32, kernel_size=23, strides=1, padding='same', activation='relu'),
        # The second pooling layer, max pooling, four 3x1 convolution kernels, with a stride of 2
        tf.keras.layers.MaxPool1D(pool_size=3, strides=2, padding='same'),
        # The third convolution layer, 32 25x1 convolution kernels
        tf.keras.layers.Conv1D(filters=64, kernel_size=25, strides=1, padding='same', activation='relu'),
        # The third pooling layer, average pooling, four 3x1 convolution kernels, stride 2
        tf.keras.layers.AvgPool1D(pool_size=3, strides=2, padding='same'),
        # The fourth convolution layer, 64 27x1 convolution kernels
        tf.keras.layers.Conv1D(filters=128, kernel_size=27, strides=1, padding='same', activation='relu'),
        # LSTM layer, 64 nodes
        tf.keras.layers.LSTM(128, return_sequences=True),
        # Dropout layer,dropout = 0.2
        tf.keras.layers.Dropout(rate=0.2),
        # LSTM layer, 64 nodes
        tf.keras.layers.LSTM(64, return_sequences=True),
        # Flatten the layer to facilitate the processing of the fully connected layer
        tf.keras.layers.Flatten(),
        # Fully connected layer, 128 nodes
        tf.keras.layers.Dense(128, activation='relu'),
        # Dropout layer,dropout = 0.2
        tf.keras.layers.Dropout(rate=0.2),
        # Fully connected layer, 5 nodes
        tf.keras.layers.Dense(5, activation='softmax')
    ])
    return newModel

### **c. Model Training**

**Build, save and then Fit the Model.**
If we have already the saved model, then no need to build, save and fit again.

In [ ]:
# Reshape for CNN/LSTM input
X_train = x_train.reshape(-1, 300, 1)
X_test  = x_test.reshape(-1, 300, 1)

Y_train = y_train
Y_test  = y_test

print("X_train shape:", X_train.shape)
print("Y_train shape:", Y_train.shape)

In [ ]:
# TensorBoard log directory (can stay local)
logdir = os.path.join(
    project_path,
    "logs",
    datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
)

# Validation ratio
ratio = 0.2


# Load or Train Model

if os.path.exists(MODEL_PATH):

    print("Loading model from Google Drive...")
    model = tf.keras.models.load_model(MODEL_PATH)
    history = None

else:

    print("No saved model found. Training model...")

    model = buildModel()

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    model.summary()


    # TensorBoard
    tensorboard_callback = tf.keras.callbacks.TensorBoard(
        log_dir=logdir,
        histogram_freq=1
    )


    # Best-weights checkpoint
    checkpoint_filepath = MODEL_DIR + "/best_weights.weights.h5"

    model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_filepath,
        save_weights_only=True,
        monitor='val_accuracy',
        mode='max',
        save_best_only=True
    )


    callbacks = [
        tensorboard_callback,
        model_checkpoint_callback
    ]


    # Train
    history = model.fit(
        X_train,
        Y_train,
        epochs=30,
        batch_size=128,
        validation_split=ratio,
        callbacks=callbacks
    )


    # Save permanently to Drive
    model.save(MODEL_PATH)

    print("Model saved to Google Drive.")


### **d. Plotting Accuracy and Loss**
Training and Validation accuray and loss curve plotting.

In [ ]:
# plot accuracy during training
if history is not None:

    plt.title('Accuracy')
    plt.plot(history.history['accuracy'], label='train accuracy')
    plt.plot(history.history['val_accuracy'], label='val accuracy')
    plt.legend()
    plt.show()

    plt.title('Loss')
    plt.plot(history.history['loss'], label='train loss')
    plt.plot(history.history['val_loss'], label='val loss')
    plt.legend()
    plt.show()

else:
    print("Model loaded from Drive — no training history available.")

In [ ]:
# Load the TensorBoard notebook extension
%load_ext tensorboard
%tensorboard --logdir "/content/data/mitdb/logs\20260205-103804"

# **Part F: Results**


## **F1: Classification Accuracy and Confusion Matrix**
The overall classification accuracy and confusion matrix generated by the follwoing code.

**Model Testing**
* a. Using **recently trained** and Saved Model after 30 Epochs.
* b. Using the **best model** saved by *'model checkpoint'* callback.

**a. Using Recently Trained Model**

In [ ]:
# evaluate the model
train_loss, train_acc = model.evaluate(X_train, Y_train, verbose=0)
test_loss, test_acc = model.evaluate(X_test, Y_test, verbose=0)
print('Training Accuracy: %.2f, Test Accuracy: %.2f' % (train_acc*100, test_acc*100))
print('Training Loss: %.2f, Test Loss: %.2f' % (train_loss*100, test_loss*100))

**b. Using the Best Model**
We saved many checkpoints of the model. Among these checkpoints we will consider the checkpoint which has the largest validation accuracy. Then copy its path and update the model weights.

In [ ]:
# ==========================================
# Load Trained ECG Model (Safe Version)
# ==========================================

import os
import tensorflow as tf


# Google Drive paths
MODEL_PATH = "/content/drive/MyDrive/ecg_model/ecg_model.keras"
CHECKPOINT_PATH = "/content/drive/MyDrive/ecg_model/best_weights.weights.h5"


# -------------------------------
# Priority 1: Load full model
# -------------------------------

if os.path.exists(MODEL_PATH):

    print("Loading full model from Drive...")

    model = tf.keras.models.load_model(MODEL_PATH)

    print("Full model loaded.")


# -------------------------------
# Priority 2: Fallback to weights
# -------------------------------

elif os.path.exists(CHECKPOINT_PATH):

    print("Full model not found. Loading from checkpoint weights...")

    model = buildModel()

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    model.load_weights(CHECKPOINT_PATH)

    print("Checkpoint weights loaded.")


# -------------------------------
# Priority 3: Train (last resort)
# -------------------------------

else:

    print("No saved model found. Training from scratch...")

    model = buildModel()

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    history = model.fit(
        X_train,
        Y_train,
        epochs=30,
        batch_size=128,
        validation_split=0.2
    )

    model.save(MODEL_PATH)

    print("Model trained and saved.")


# -------------------------------
# Verify
# -------------------------------

model.summary()


In [ ]:
# train_loss_cp, train_acc_cp = model2.evaluate(X_train, Y_train, verbose=0)
# test_loss_cp, test_acc_cp = model2.evaluate(X_test, Y_test, verbose=0)
# print('Training Accuracy: %.2f, Test Accuracy: %.2f' % (train_acc_cp*100, test_acc_cp*100))
# print('Training Loss: %.2f, Test Loss: %.2f' % (train_loss_cp*100, test_loss_cp*100))
# Fast evaluation of best model (use large batch size)

# ==========================================
# Evaluate Loaded Model (Production Model)
# ==========================================

# Evaluate on TEST set only
test_loss, test_acc = model.evaluate(
    X_test,
    Y_test,
    batch_size=256,
    verbose=1
)

print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")


**Confusion Matrix**

In [ ]:
# confusion matrix
def plotHeatMap(Y_test, Y_pred):
    con_mat = confusion_matrix(Y_test, Y_pred)
    # Normalized
    # con_mat_norm = con_mat.astype('float') / con_mat.sum(axis=1)[:, np.newaxis]
    # con_mat_norm = np.around(con_mat_norm, decimals=2)

    # Plotting
    plt.figure(figsize=(8, 8))
    seaborn.heatmap(con_mat, annot=True, square=True, fmt='.20g', cmap='Greens')
    plt.ylim(0, 5)
    plt.xlabel('Predicted labels')
    plt.ylabel('True labels')
    plt.show()

In [ ]:
# predict
Y_pred = np.argmax(model.predict(X_test), axis=-1)
# Y_pred = model.predict_classes(X_test)
# plot confusion matrix
plotHeatMap(Y_test, Y_pred)

## **F2: Per Class Performance**
Calculating per class Sensitivity, Specificity, Accuracy and F1 score.

In [ ]:
# Per class accuracy printing function
def _report(TN, FP, FN, TP):
    TPR = TP/(TP+FN) if (TP+FN)!=0 else 0
    TNR = TN/(TN+FP) if (TN+FP)!=0 else 0
    PPV = TP/(TP+FP) if (TP+FP)!=0 else 0
    '''
    report = {'TP': TP, 'TN': TN, 'FP': FP, 'FN': FN,
              'TPR': TPR, 'Recall': TPR, 'Sensitivity': TPR,
              'TNR' : TNR, 'Specificity': TNR,
              'FPR': FP/(FP+TN) if (FP+TN)!=0 else 0,
              'FNR': FN/(FN+TP) if (FN+TP)!=0 else 0,
              'PPV': PPV, 'Precision': PPV,
              'F1 Score': 2*(PPV*TPR)/(PPV+TPR),
              'Per Class Accuracy': (TP+TN)/(TP+FP+FN+TN)
             }'''

    report = {'Sensitivity (%)': TPR*100,
              'Specificity (%)': TNR*100,
              'F1 Score (%)': 2*100*(PPV*TPR)/(PPV+TPR),
              'Per Class Accuracy (%)': (TP+TN)*100/(TP+FP+FN+TN)
             }
    return report

def multi_classification_report(y_test, y_pred, labels=None, encoded_labels=True, as_frame=False):
    """
    Args:
        y_test (ndarray)
        y_pred (ndarray)
        labels (list)
        encoded_labels (bool): Need to be False if labels are not one hot encoded
        as_fram (bool): If True, return type will be DataFrame

    Return:
        report (dict)
    """

    import numpy as np
    import pandas as pd
    from sklearn.metrics import multilabel_confusion_matrix

    conf_labels = None if encoded_labels else labels

    conf_mat = multilabel_confusion_matrix(y_test, y_pred, labels=conf_labels)
    report = dict()
    if labels == None:
        counter = np.arange(len(conf_mat))
    else:
        counter = labels

    for i, name in enumerate(counter):
        TN, FP, FN, TP = conf_mat[i].ravel()
        report[name] = _report(TN, FP, FN, TP)

    if as_frame:
        return pd.DataFrame(report)
    return report

**a. Recently Trained Model**

In [ ]:
# Per class performance
labels = ['N', 'S', 'V', 'F', 'Q']
Y_pred = np.argmax(model.predict(X_test), axis=-1)
multi_classification_report(Y_test, Y_pred, labels=labels, encoded_labels=True, as_frame=True)

**b. Best Model**

In [ ]:
# Per class performance
# Per class performance (Loaded Production Model)
labels = ['N', 'S', 'V', 'F', 'Q']

Y_pred = np.argmax(model.predict(X_test), axis=-1)

multi_classification_report(
    Y_test,
    Y_pred,
    labels=labels,
    encoded_labels=True,
    as_frame=True
)

## **F3: Interactive Patient ECG Viewer**


In [ ]:
# ==========================================
# F3 — Interactive ECG Viewer (Final Version)
# ==========================================

import wfdb
import neurokit2 as nk
import plotly.graph_objects as go
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import warnings


# -------------------------------
# Suppress Pandas / NeuroKit noise
# -------------------------------

warnings.filterwarnings("ignore", message=".*ChainedAssignment.*")
warnings.filterwarnings("ignore", message=".*copy of a DataFrame.*")


# -------------------------------
# Load example patient
# -------------------------------

record = wfdb.rdrecord(project_path + "100", channel_names=["MLII"])
data = record.p_signal.flatten()


# -------------------------------
# Denoise (your existing function)
# -------------------------------

rdata = denoise(data)


# -------------------------------
# Normalise (improves detection)
# -------------------------------

scaler = MinMaxScaler()
rdata = scaler.fit_transform(
    rdata.reshape(-1, 1)
).flatten()


# -------------------------------
# Trim to first 30s (speed)
# -------------------------------

fs = 360
duration = 30  # seconds

rdata = rdata[: fs * duration]


# -------------------------------
# NeuroKit Processing
# -------------------------------

signals, info = nk.ecg_process(
    rdata,
    sampling_rate=fs,
    method="neurokit"
)


# Convert to NumPy (safe for Pandas 3.x)
p_peaks = signals["ECG_P_Peaks"].to_numpy()
r_peaks = signals["ECG_R_Peaks"].to_numpy()
t_peaks = signals["ECG_T_Peaks"].to_numpy()


# -------------------------------
# Time axis
# -------------------------------

time = np.arange(len(rdata)) / fs


# -------------------------------
# Interactive Plot
# -------------------------------

fig = go.Figure()


# ECG waveform
fig.add_trace(
    go.Scatter(
        x=time,
        y=rdata,
        name="ECG",
        line=dict(color="black", width=1),
        hovertemplate="Time: %{x:.2f}s<br>Voltage: %{y:.3f}"
    )
)


# P waves
fig.add_trace(
    go.Scatter(
        x=time[p_peaks == 1],
        y=rdata[p_peaks == 1],
        mode="markers",
        name="P wave",
        marker=dict(size=6, color="blue")
    )
)


# R peaks
fig.add_trace(
    go.Scatter(
        x=time[r_peaks == 1],
        y=rdata[r_peaks == 1],
        mode="markers",
        name="R peak",
        marker=dict(size=8, color="green")
    )
)


# T waves
fig.add_trace(
    go.Scatter(
        x=time[t_peaks == 1],
        y=rdata[t_peaks == 1],
        mode="markers",
        name="T wave",
        marker=dict(size=6, color="orange")
    )
)


# -------------------------------
# Layout
# -------------------------------

fig.update_layout(
    title="Interactive ECG With P/QRS/T Annotations",
    xaxis_title="Time (seconds)",
    yaxis_title="Normalised Voltage",
    template="simple_white",
    height=600,
    legend=dict(orientation="h"),
    hovermode="x unified"
)


# -------------------------------
# Save for GitHub / Dissertation
# -------------------------------

# Save interactive version (works everywhere)
fig.write_html(
    "/content/drive/MyDrive/ecg_model/ecg_viewer.html"
)

print("Saved interactive ECG viewer to Google Drive.")


# -------------------------------
# Show in Colab
# -------------------------------

fig.show()

# **Part G: Calibration (temperature scaling + ECE)**


In [ ]:
# Create logits model (model output before softmax)
from tensorflow.keras.models import Model
import numpy as np

logits_model = Model(
    inputs=model.input,
    outputs=model.layers[-1].input
)

print("Logits model created")

In [ ]:
from scipy.optimize import minimize

def temperature_scale(T, logits, labels):
    logits_T = logits / T
    exp = np.exp(logits_T)
    probs = exp / np.sum(exp, axis=1, keepdims=True)
    log_likelihood = -np.log(probs[np.arange(len(labels)), labels])
    return np.mean(log_likelihood)

# Compute logits
logits_val = logits_model.predict(x_train)

# Optimise temperature
opt = minimize(
    temperature_scale,
    x0=[1.0],
    args=(logits_val, y_train),
    bounds=[(0.05, 10)]
)

T_star = opt.x[0]

print("Optimal Temperature:", T_star)

In [ ]:
def predict_proba_calibrated(x):

    logits = logits_model.predict(x)
    logits = logits / T_star

    exp = np.exp(logits)
    probs = exp / np.sum(exp, axis=1, keepdims=True)

    return probs

In [ ]:
def compute_ece(probs, labels, n_bins=15):

    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracy = predictions == labels

    bins = np.linspace(0,1,n_bins+1)

    ece = 0
    for i in range(n_bins):

        mask = (confidences >= bins[i]) & (confidences < bins[i+1])

        if np.sum(mask) > 0:
            acc = np.mean(accuracy[mask])
            conf = np.mean(confidences[mask])

            ece += np.abs(acc-conf)*np.sum(mask)/len(labels)

    return ece

probs_uncal = model.predict(x_test)
probs_cal = predict_proba_calibrated(x_test)

ece_uncal = compute_ece(probs_uncal, y_test)
ece_cal = compute_ece(probs_cal, y_test)

print("ECE before:", ece_uncal)
print("ECE after:", ece_cal)

# **Part H: Conformal Prediction (Uncertainty Sets)**


In [ ]:
alpha = 0.1  # 90% coverage

# Split calibration set
from sklearn.model_selection import train_test_split

X_fit, X_cal, y_fit, y_cal = train_test_split(
    x_train,
    y_train,
    test_size=0.2,
    stratify=y_train
)

# calibrated probabilities
cal_probs = predict_proba_calibrated(X_cal)

scores = 1 - cal_probs[np.arange(len(y_cal)), y_cal]

qhat = np.quantile(scores, 1-alpha)

print("Conformal threshold qhat:", qhat)

In [ ]:
def conformal_prediction_set(x):

    probs = predict_proba_calibrated(x)

    prediction_sets = []

    for p in probs:

        scores = 1 - p

        prediction_sets.append(
            np.where(scores <= qhat)[0]
        )

    return prediction_sets

# **Part I: Grad-CAM Explainability**

In [ ]:
import tensorflow as tf

last_conv = None

for layer in reversed(model.layers):
    if isinstance(layer, tf.keras.layers.Conv1D):
        last_conv = layer.name
        break

grad_model = tf.keras.models.Model(
    inputs=model.input,
    outputs=[model.get_layer(last_conv).output, model.output]
)

def gradcam_1d(signal):

    signal = signal[np.newaxis,...]

    with tf.GradientTape() as tape:

        conv_out, preds = grad_model(signal)
        class_idx = tf.argmax(preds[0])

        loss = preds[:,class_idx]

    grads = tape.gradient(loss, conv_out)

    weights = tf.reduce_mean(grads, axis=1)

    cam = tf.reduce_sum(weights[:, :, None] * conv_out, axis=-1)

    cam = tf.nn.relu(cam)

    cam = cam / tf.reduce_max(cam)

    cam = cam.numpy()[0]

    cam = np.interp(
        np.linspace(0,len(cam)-1,300),
        np.arange(len(cam)),
        cam
    )

    return cam

# **Part J: Prototype Retrieval (Example Explanations)**

In [ ]:
embedding_model = Model(
    inputs=model.input,
    outputs=model.layers[-2].output
)

X_bank = X_fit[:5000]
y_bank = y_fit[:5000]

emb_bank = embedding_model.predict(X_bank)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_topk(signal, k=3):

    emb = embedding_model.predict(signal[np.newaxis,...])

    sim = cosine_similarity(emb, emb_bank)[0]

    idx = np.argsort(sim)[-k:][::-1]

    return idx, X_bank[idx], y_bank[idx]

# **Part K: Robustness Testing (NSTDB Noise)**

In [ ]:
def add_noise(signal, noise, snr_db):

    signal_power = np.mean(signal**2)

    noise_power = np.mean(noise**2)

    factor = np.sqrt(signal_power/(noise_power*(10**(snr_db/10))))

    return signal + noise*factor

In [ ]:
snr_levels = [24,18,12,6,0]

accs = []

noise = np.random.normal(0,1,300)

for snr in snr_levels:

    X_noisy = np.array([
        add_noise(x.squeeze(),noise,snr)
        for x in x_test[:2000]
    ])

    X_noisy = X_noisy[...,None]

    preds = np.argmax(
        predict_proba_calibrated(X_noisy),
        axis=1
    )

    acc = np.mean(preds == y_test[:2000])

    accs.append(acc)

import matplotlib.pyplot as plt

plt.plot(snr_levels, accs)
plt.xlabel("SNR")
plt.ylabel("Accuracy")
plt.title("Noise Robustness")
plt.show()

# **Part L: Prediction Engine**

In [ ]:
# ==========================================
# Prediction Engine
# ==========================================

CLASS_NAMES = ["Normal", "Supraventricular", "Ventricular", "Fusion", "Unknown"]

def predict_heartbeat(ecg_segment):

    ecg_segment = ecg_segment.reshape(1,300,1)

    # Use calibrated probabilities
    probs = predict_proba_calibrated(ecg_segment)[0]

    predicted_class = np.argmax(probs)

    result = {
        "class_index": int(predicted_class),
        "class_name": CLASS_NAMES[predicted_class],
        "probabilities": probs.tolist()
    }

    return result

# **Part M: Confidence Module**

In [ ]:
# ==========================================
# Confidence Metrics
# ==========================================

from scipy.stats import entropy
import numpy as np

def prediction_confidence(probabilities):

    probabilities = np.array(probabilities)

    max_prob = np.max(probabilities)

    sorted_probs = np.sort(probabilities)

    margin = sorted_probs[-1] - sorted_probs[-2]

    uncertainty = entropy(probabilities)

    confidence = {
        "confidence_score": float(max_prob),
        "prediction_margin": float(margin),
        "uncertainty_entropy": float(uncertainty)
    }

    return confidence

# **Part N: Risk Indicator**

In [ ]:
# ==========================================
# Risk Scoring
# ==========================================

RISK_MAP = {
    "Normal": "Low",
    "Supraventricular": "Moderate",
    "Ventricular": "High",
    "Fusion": "Moderate",
    "Unknown": "High"
}

def risk_indicator(class_name, confidence, conformal_size):

    risk_level = RISK_MAP[class_name]

    if confidence < 0.60:
        risk_level = "Uncertain"

    if conformal_size > 1:
        risk_level = "Ambiguous"

    return risk_level

# **Part O: Inference Pipeline**

In [ ]:
# ==========================================
# Full ECG Inference Pipeline
# ==========================================

def ecg_inference(ecg_segment):

    prediction = predict_heartbeat(ecg_segment)

    confidence = prediction_confidence(prediction["probabilities"])

    conformal = conformal_prediction_set(
        ecg_segment.reshape(1,300,1)
    )[0]

    conformal_names = [CLASS_NAMES[i] for i in conformal]

    risk = risk_indicator(
        prediction["class_name"],
        confidence["confidence_score"],
        len(conformal)
    )

    # Explainability modules
    cam = gradcam_1d(ecg_segment)

    idx, proto_x, proto_y = retrieve_topk(ecg_segment)

    result = {

        "prediction": prediction["class_name"],

        "confidence": confidence["confidence_score"],

        "risk_level": risk,

        "conformal_set": conformal_names,

        "details": {
            "probabilities": prediction["probabilities"],
            "margin": confidence["prediction_margin"],
            "entropy": confidence["uncertainty_entropy"]
        },

        "explanations": {
            "gradcam": cam.tolist(),
            "prototype_labels": proto_y.tolist()
        }
    }

    return result

# **Part P: Example Prediction**

In [ ]:
import numpy as np

for i in np.random.randint(0, len(X_test), 10):

    sample = X_test[i].flatten()

    result = ecg_inference(sample)

    true_label = CLASS_NAMES[int(Y_test[i])]

    print(
        f"True: {true_label} | "
        f"Pred: {result['prediction']} | "
        f"Conf: {result['confidence']:.4f} | "
        f"Risk: {result['risk_level']}"
    )